In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# 0 = all messages are logged (default behavior)
# 1 = INFO messages are not printed
# 2 = INFO and WARNING messages are not printed
# 3 = INFO, WARNING, and ERROR messages are not printed

import datetime

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
#from google.colab import drive
#drive.mount('/content/gdrive')

mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

#import warnings
# https://stackoverflow.com/questions/15777951/how-to-suppress-pandas-future-warning
#warnings.simplefilter(action='ignore', category=FutureWarning)
#warnings.simplefilter(action='ignore', category=Warning)

tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import logging
tf.get_logger().setLevel(logging.ERROR)

# https://stackoverflow.com/questions/65697623/tensorflow-warning-found-untraced-functions-such-as-lstm-cell-6-layer-call-and
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from keras.models import load_model

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Sep  2 17:46:41 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-SXM...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   34C    P0    42W / 400W |      0MiB / 40960MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [3]:
# Mounting Drive
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
%cd /content/drive/My Drive/TX_DATA/BestData

/content/drive/My Drive/TX_DATA/BestData


In [5]:
#Read In Data
dfs = {}
for index in range(0, 6):
  df = pd.read_csv('Station' + str(index + 1) + '-Simulated-cleaned-merged-data.csv', sep=",", parse_dates=["Date"], index_col="Date")
  dfs['Station' + str(index + 1)] = df

# Vectorize wind
for station, df in dfs.items():
  wv = df.pop('Windspeed')
  wd_rad = df.pop('Winddirection')*np.pi / 180
  df['Wx'] = wv*np.cos(wd_rad)
  df['Wy'] = wv*np.sin(wd_rad)
  dfs[station] = df

# Remove periodicity in time data (remove daily and yearly periodicity)
day = 24*60*60
year = (365.2425)*day

for station, df in dfs.items() :
  timestamp_s = (df.index).map(pd.Timestamp.timestamp)
  df['Day sin'] = np.sin(timestamp_s * (2 * np.pi / day))
  df['Day cos'] = np.cos(timestamp_s * (2 * np.pi / day))
  df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
  df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))
  dfs[station] = df

for station, df in dfs.items():
  lat = df.pop('Latitude')
  lon = df.pop('Longitude')
  df["x_cord"] = np.cos(lat) * np.cos(lon)
  df["y_cord"] = np.sin(lat) * np.cos(lon)
  df["z_cord"] = np.sin(lon)
  dfs[station] = df





In [6]:
dfs["Station1"].head()

,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Tair,RH,...,Ppt,Wx,Wy,Day sin,Day cos,Year sin,Year cos,x_cord,y_cord,z_cord
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-01 00:00:00,0.139,0.178,0.148,0.152,2.81,4.40,5.77,10.57,-1.090,81.5,...,0.0,0.643762,0.832030,1.232357e-12,1.000000,0.001505,0.999999,-0.180165,0.291386,0.939486
2015-01-01 01:00:00,0.139,0.178,0.148,0.152,2.86,4.38,5.71,10.51,-1.038,81.7,...,0.0,0.657578,0.698049,2.588190e-01,0.965926,0.002222,0.999998,-0.180165,0.291386,0.939486
2015-01-01 02:00:00,0.139,0.178,0.148,0.152,2.89,4.35,5.66,10.47,-0.981,82.0,...,0.0,0.653248,0.837324,5.000000e-01,0.866025,0.002939,0.999996,-0.180165,0.291386,0.939486
2015-01-01 03:00:00,0.139,0.178,0.148,0.152,2.90,4.33,5.62,10.41,-0.814,81.9,...,0.0,0.458032,0.759589,7.071068e-01,0.707107,0.003656,0.999993,-0.180165,0.291386,0.939486
2015-01-01 04:00:00,0.139,0.178,0.148,0.152,2.96,4.32,5.59,10.36,-0.805,90.0,...,0.0,0.793697,0.235857,8.660254e-01,0.500000,0.004372,0.999990,-0.180165,0.291386,0.939486


In [7]:
#Use only overlapping indexes
index_union = pd.Index([])
for station, df in dfs.items():
  index_union = index_union.union(df.index)
index_int = index_union
for station, df in dfs.items():
  index_int = index_int.intersection(df.index)

In [8]:
for key in dfs.keys():
  dfs[key] = dfs[key].loc[index_int]

In [9]:
for key, df in dfs.items():
  print(key, df.shape)

Station1 (55028, 21)
Station2 (55028, 21)
Station3 (55028, 21)
Station4 (55028, 21)
Station5 (55028, 21)
Station6 (55028, 21)


In [10]:
s1 = dfs["Station1"].values
s2 = dfs["Station2"].values
s3 = dfs["Station3"].values
s4 = dfs["Station4"].values
s5 = dfs["Station5"].values
s6 = dfs["Station6"].values

cur_df = pd.DataFrame(np.vstack((s1,s2,s3,s4,s5,s6)))

cur_df.columns = dfs["Station1"].columns

cur_df

,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Tair,RH,...,Ppt,Wx,Wy,Day sin,Day cos,Year sin,Year cos,x_cord,y_cord,z_cord
0,0.139,0.178,0.148,0.152,2.81,4.40,5.77,10.57,-1.090,81.50,...,0.0,0.643762,0.832030,1.232357e-12,1.000000,0.001505,0.999999,-0.180165,0.291386,0.939486
1,0.139,0.178,0.148,0.152,2.86,4.38,5.71,10.51,-1.038,81.70,...,0.0,0.657578,0.698049,2.588190e-01,0.965926,0.002222,0.999998,-0.180165,0.291386,0.939486
2,0.139,0.178,0.148,0.152,2.89,4.35,5.66,10.47,-0.981,82.00,...,0.0,0.653248,0.837324,5.000000e-01,0.866025,0.002939,0.999996,-0.180165,0.291386,0.939486
3,0.139,0.178,0.148,0.152,2.90,4.33,5.62,10.41,-0.814,81.90,...,0.0,0.458032,0.759589,7.071068e-01,0.707107,0.003656,0.999993,-0.180165,0.291386,0.939486
4,0.139,0.178,0.148,0.152,2.96,4.32,5.59,10.36,-0.805,90.00,...,0.0,0.793697,0.235857,8.660254e-01,0.500000,0.004372,0.999990,-0.180165,0.291386,0.939486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330163,0.180,0.175,0.170,0.111,22.74,22.90,23.17,23.73,24.420,89.30,...,0.0,-2.523052,-0.309792,7.071068e-01,-0.707107,0.426507,-0.904484,-0.097600,0.212437,0.972288
330164,0.180,0.175,0.170,0.111,23.06,23.10,23.19,23.69,25.770,75.79,...,0.0,-1.975161,0.518174,5.000000e-01,-0.866025,0.425859,-0.904790,-0.097600,0.212437,0.972288
330165,0.180,0.175,0.169,0.111,23.79,23.55,23.30,23.65,28.180,68.67,...,0.0,-2.993172,0.394059,2.588190e-01,-0.965926,0.425210,-0.905095,-0.097600,0.212437,0.972288
330166,0.180,0.175,0.169,0.111,25.15,24.47,23.62,23.61,29.730,65.87,...,0.0,-3.030859,-0.313209,9.954815e-12,-1.000000,0.424561,-0.905399,-0.097600,0.212437,0.972288


In [11]:
wx = cur_df.pop("Wx")
wy = cur_df.pop("Wy")
d_sin = cur_df.pop("Day sin")
d_cos = cur_df.pop("Day cos")
y_sin = cur_df.pop("Year sin")
y_cos = cur_df.pop("Year cos")
x = cur_df.pop("x_cord")
y = cur_df.pop("y_cord")
z = cur_df.pop("z_cord")

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled_df = pd.DataFrame(data = scaler.fit_transform(cur_df), columns = cur_df.columns)

scaled_df["Wx"] = wx
scaled_df["Wy"] = wy
scaled_df["Day sin"] = d_sin
scaled_df["Day cos"] = d_cos
scaled_df["Year sin"] = y_sin
scaled_df["Year cos"] = y_cos
scaled_df["x_cord"] = x
scaled_df["y_cord"] = y
scaled_df["z_cord"] = z

scaled_df

,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Tair,RH,...,Ppt,Wx,Wy,Day sin,Day cos,Year sin,Year cos,x_cord,y_cord,z_cord
0,0.240602,0.300752,0.143050,0.222707,0.047667,0.072934,0.140049,0.200591,0.796769,0.905756,...,0.0,0.643762,0.832030,1.232357e-12,1.000000,0.001505,0.999999,-0.180165,0.291386,0.939486
1,0.240602,0.300752,0.143050,0.222707,0.048677,0.072470,0.138592,0.198777,0.797009,0.906775,...,0.0,0.657578,0.698049,2.588190e-01,0.965926,0.002222,0.999998,-0.180165,0.291386,0.939486
2,0.240602,0.300752,0.143050,0.222707,0.049283,0.071776,0.137379,0.197567,0.797273,0.908304,...,0.0,0.653248,0.837324,5.000000e-01,0.866025,0.002939,0.999996,-0.180165,0.291386,0.939486
3,0.240602,0.300752,0.143050,0.222707,0.049485,0.071313,0.136408,0.195753,0.798046,0.907794,...,0.0,0.458032,0.759589,7.071068e-01,0.707107,0.003656,0.999993,-0.180165,0.291386,0.939486
4,0.240602,0.300752,0.143050,0.222707,0.050697,0.071081,0.135680,0.194241,0.798088,0.949058,...,0.0,0.793697,0.235857,8.660254e-01,0.500000,0.004372,0.999990,-0.180165,0.291386,0.939486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330163,0.343358,0.293233,0.183397,0.133188,0.450212,0.501273,0.562379,0.598546,0.914865,0.945492,...,0.0,-2.523052,-0.309792,7.071068e-01,-0.707107,0.426507,-0.904484,-0.097600,0.212437,0.972288
330164,0.343358,0.293233,0.183397,0.133188,0.456675,0.505904,0.562864,0.597337,0.921115,0.876668,...,0.0,-1.975161,0.518174,5.000000e-01,-0.866025,0.425859,-0.904790,-0.097600,0.212437,0.972288
330165,0.343358,0.293233,0.181563,0.133188,0.471420,0.516323,0.565534,0.596127,0.932272,0.840397,...,0.0,-2.993172,0.394059,2.588190e-01,-0.965926,0.425210,-0.905095,-0.097600,0.212437,0.972288
330166,0.343358,0.293233,0.181563,0.133188,0.498889,0.537624,0.573301,0.594917,0.939447,0.826133,...,0.0,-3.030859,-0.313209,9.954815e-12,-1.000000,0.424561,-0.905399,-0.097600,0.212437,0.972288


In [12]:
#Definitions
TARGET_COL = "SWC_5"
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.2
WINDOW_SIZE = 24*7*3
SHIFT_AMT = 24*7
PAT = 3

In [13]:
def split_and_scale(df, target_col=TARGET_COL, train_split = TRAIN_SPLIT, val_split = VAL_SPLIT):
  print(df.columns)
  target_idx = df.columns.get_loc(target_col)
  train_set = df[ : int(len(df) * train_split)].values
  val_set = df[int(len(df) * train_split) : int(len(df) * (train_split + val_split))].values
  test_set = df[int(len(df) * (train_split + val_split)) :].values

  #train_set = scaler.fit_transform(train_set)
  #val_set = scaler.transform(val_set)
  #test_set = scaler.transform(test_set)

  return (train_set, val_set, test_set, target_idx)

def generate_windows(data, window_size=24, shift=24, target_idx=0):
    labels = data[:, target_idx]

    X = []
    y = []
    for i in range(len(data) - window_size - shift):
        # get window based on input width
        window = data[i : i + window_size]

        # keep track of label associated with current window
        window_label = labels[i + window_size + shift]

        X.append(window)
        y.append(window_label)

    # in new dataset, each element is a data window, and window label is single value
    return np.array(X), np.array(y)


# given data and its labels, divide the data into batches of size batch_size
def generate_batches(X, y, batch_size=64):
    # divides data into batches, drops any remainder batches smaller than specified batch size.
    # allows models to run with any batch size
    tf_dataset = tf.data.Dataset.from_tensor_slices((X, y))
    tf_dataset = tf_dataset.repeat().batch(batch_size=batch_size, drop_remainder=True)

    # tf_dataset repeats indefinitely, need to compute number of step updates to complete 1 epoch
    steps_per_epoch = len(X) // batch_size

    return (tf_dataset, steps_per_epoch)

def preprocess_data(df):
    # data cleaning and feature engineering
    print("1")
    train_set, val_set, test_set, target_idx = split_and_scale(df)

    print("2")
    # create window data for each dataset
    X_train, y_train = generate_windows(train_set, window_size=WINDOW_SIZE, shift=SHIFT_AMT, target_idx=target_idx)
    X_val, y_val = generate_windows(val_set, window_size=WINDOW_SIZE, shift=SHIFT_AMT, target_idx=target_idx)
    X_test, y_test = generate_windows(test_set, window_size=WINDOW_SIZE, shift=SHIFT_AMT, target_idx=target_idx)

    return (X_train, y_train, X_val, y_val, X_test, y_test)

In [14]:
BATCH_SIZE = 4096
train_set, val_set, test_set, target_idx = split_and_scale(scaled_df)

Index(['SWC_5', 'SWC_10', 'SWC_20', 'SWC_50', 'T_5', 'T_10', 'T_20', 'T_50',
       'Tair', 'RH', 'Srad', 'Ppt', 'Wx', 'Wy', 'Day sin', 'Day cos',
       'Year sin', 'Year cos', 'x_cord', 'y_cord', 'z_cord'],
      dtype='object')


In [15]:
X_train, y_train = generate_windows(train_set, window_size=WINDOW_SIZE, shift=SHIFT_AMT, target_idx=target_idx)

In [16]:
X_val, y_val = generate_windows(val_set, window_size=WINDOW_SIZE, shift=SHIFT_AMT, target_idx=target_idx)

In [17]:
X_test, y_test = generate_windows(test_set, window_size=WINDOW_SIZE, shift=SHIFT_AMT, target_idx=target_idx)

In [18]:
train_dataset, train_steps = generate_batches(X_train, y_train, batch_size=BATCH_SIZE)

In [19]:
val_dataset, val_steps = generate_batches(X_val, y_val, batch_size=BATCH_SIZE)

In [20]:
test_dataset, test_steps = generate_batches(X_test, y_test, batch_size=BATCH_SIZE)

In [21]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(230445, 504, 21)
(65362, 504, 21)
(32345, 504, 21)


In [22]:
MAX_EPOCHS = 5

def compile_and_fit(model, data, steps_per_epoch, val_data, val_steps, model_name='model/', patience=3, max_epochs=MAX_EPOCHS, batch_size=32):
    # stop running epochs if the loss stops improving for patience number of epochs
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, mode='min')

    # store the best model on disk to be loaded later without having to re-fit
    # allows you to load models from disc
    ckpt = tf.keras.callbacks.ModelCheckpoint(model_name, save_best_only=True)

    model.compile(loss=tf.keras.losses.MeanSquaredError(),
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=[tf.keras.metrics.MeanAbsoluteError(), tf.keras.metrics.MeanSquaredError()])

    history = model.fit(data,
                        epochs=max_epochs,
                        callbacks=[ckpt, early_stopping],
                        validation_data=val_data,
                        validation_steps=val_steps,
                        shuffle=False,
                        batch_size=batch_size,
                        steps_per_epoch = steps_per_epoch)

    return history

In [23]:
def plot_single_pred(model, dataset, data_steps, y, batch_size=32):
    forecast = model.predict(dataset, batch_size=batch_size, steps=data_steps)

    if len(forecast.shape) == 3:
        print("asd")
        forecast = forecast[:, 0, 0]
    elif len(forecast.shape) == 2:
        forecast = forecast[:, 0]

    plt.figure(figsize=(10, 6))
    plot_data = {"Predictions": forecast, "Actual": y}

    plt.plot(plot_data["Actual"])
    plt.plot(plot_data["Predictions"])

    plt.legend(("Actual", "Predictions"))

    return plot_data

In [24]:
#maybe remove one relu dense layer
lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.LSTM(64, return_sequences=False),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=16, activation='relu'),
    tf.keras.layers.Dense(units=1, activation='tanh')
])

In [25]:
loss_by_epoch = {}
val_performance = {}
performance = {}

In [ ]:
history = compile_and_fit(lstm_model, train_dataset, train_steps, val_dataset, val_steps, batch_size=BATCH_SIZE, model_name="RNN", patience=PAT)
loss_by_epoch["RNN"] = history.history
val_performance["RNN"] = lstm_model.evaluate(val_dataset, steps=val_steps, batch_size=BATCH_SIZE, verbose=1)
performance["RNN"] = lstm_model.evaluate(test_dataset, steps=test_steps, batch_size=BATCH_SIZE, verbose=0)

Epoch 1/5
56/56 [==============================] - 43s 570ms/step - loss: 0.0406 - mean_absolute_error: 0.1631 - mean_squared_error: 0.0406 - val_loss: 0.0483 - val_mean_absolute_error: 0.1803 - val_mean_squared_error: 0.0483
Epoch 2/5
56/56 [==============================] - ETA: 0s - loss: 0.0225 - mean_absolute_error: 0.1224 - mean_squared_error: 0.0225

In [ ]:
performance["RNN"][1]

In [ ]:
plot_single_pred(lstm_model, test_dataset, test_steps, y_test, batch_size=BATCH_SIZE)